# Module 5: Biosecurity, Ethics & The Future
## Lab: From Capstone to Pitch — Responsible Bio-Design

**Lagos Bio-Design Bootcamp** | [JobAiReady](https://github.com/JobAiReady/lagos-bio-design)

---

### Objective
Analyse the biosecurity implications of your capstone protein designs, apply responsible screening practices, and compile your results into a startup pitch.

### Prerequisites
- Completed Modules 1–4 (especially the Capstone)
- No GPU required — CPU runtime is fine

### Deliverable
A 5-minute pitch deck (PDF or slides) covering your protein design, its impact, and your biosecurity analysis.

### Context
As generative protein design becomes more accessible, the field faces a dual-use dilemma: the same tools that design diagnostics can theoretically design threats. This module teaches you to think critically about these risks and build responsible practices into your workflow.

> ⚠️ **This module does NOT require GPU.** CPU runtime is sufficient.

---
## Setup

In [ ]:
!pip install -q biopython matplotlib numpy pandas

import os
import json
import hashlib
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

print('Module 5 environment ready.')
print(f'Date: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}')

---
## Part 1: The FoldMark Concept — Auditing AI-Designed Proteins

The **FoldMark** paper proposes that AI-generated proteins should carry digital signatures so they can be traced back to their origin model and designer. This is analogous to watermarking in AI-generated images.

### Why It Matters
- **Traceability**: If a designed protein is found in a harmful context, who made it?
- **Accountability**: Labs and models should be auditable.
- **Trust**: Regulatory bodies need confidence that designs are traceable.

Let's implement a simple design audit trail for your capstone sequences.

In [ ]:
def create_design_audit(sequence, designer, model_used, purpose, target_organism=None):
    """Create an audit record for a designed protein sequence."""
    # Generate a unique fingerprint from the sequence
    seq_hash = hashlib.sha256(sequence.encode()).hexdigest()[:16]
    
    audit = {
        'sequence_id': f'LBD-{seq_hash}',
        'sequence_hash': hashlib.sha256(sequence.encode()).hexdigest(),
        'length': len(sequence),
        'designer': designer,
        'affiliation': 'Lagos Bio-Design Bootcamp, Cohort 1',
        'model_used': model_used,
        'purpose': purpose,
        'target_organism': target_organism,
        'timestamp': datetime.datetime.now().isoformat(),
        'dual_use_reviewed': False,  # Must be set to True after review
        'ethics_approved': False      # Must be set to True after review
    }
    return audit

# Example: audit a capstone binder sequence
example_seq = 'MKTLLILAVFCLASQCGAELREACKDGELSCADKSKGKCEAKNKKECPSVFNETCEPCR'

audit = create_design_audit(
    sequence=example_seq,
    designer='Student Name',
    model_used='RFDiffusion + ProteinMPNN',
    purpose='Diagnostic binder for Lassa GPC (PDB: 5VK2)',
    target_organism='Lassa mammarenavirus'
)

print('=== DESIGN AUDIT RECORD ===')
for k, v in audit.items():
    print(f'  {k}: {v}')

In [ ]:
def detect_design_signature(sequence):
    """Heuristic analysis: does this sequence look AI-designed vs natural?
    
    AI-designed proteins often show distinctive amino acid distributions
    compared to natural proteins. This is a simplified demonstration.
    """
    # Natural amino acid frequencies (approximate, from UniProt)
    natural_freq = {
        'A': 8.25, 'R': 5.53, 'N': 4.06, 'D': 5.45, 'C': 1.37,
        'Q': 3.93, 'E': 6.75, 'G': 7.07, 'H': 2.27, 'I': 5.96,
        'L': 9.66, 'K': 5.84, 'M': 2.42, 'F': 3.86, 'P': 4.70,
        'S': 6.56, 'T': 5.34, 'W': 1.08, 'Y': 2.92, 'V': 6.87
    }
    
    # Calculate sequence frequencies
    total = len(sequence)
    seq_counts = Counter(sequence)
    seq_freq = {aa: (seq_counts.get(aa, 0) / total) * 100 for aa in natural_freq}
    
    # Chi-squared-like deviation from natural
    deviation = sum(
        ((seq_freq[aa] - natural_freq[aa]) ** 2) / natural_freq[aa]
        for aa in natural_freq
    )
    
    # High deviation suggests non-natural composition
    is_likely_designed = deviation > 15  # Empirical threshold
    
    return {
        'deviation_score': round(deviation, 2),
        'likely_designed': is_likely_designed,
        'assessment': 'Likely AI-designed' if is_likely_designed else 'Consistent with natural protein'
    }

# Test with our example
result = detect_design_signature(example_seq)
print(f'Deviation score: {result["deviation_score"]}')
print(f'Assessment: {result["assessment"]}')
print()
print('Note: This is a simplified heuristic. Real FoldMark approaches use')
print('deep learning classifiers trained on millions of natural vs designed sequences.')

---
## Part 2: Biosecurity Screening — The Dual-Use Dilemma

### Key Question
**Can the same tools that design a Lassa diagnostic also design a bioweapon?**

The answer is nuanced. Sequence-level screening against known threat agents is a first-line defence.

### Real-World Practice
Companies like Twist Bioscience and IDT screen every DNA synthesis order against:
- **Regulated Pathogen Lists** (CDC Select Agents, Australia Group)
- **Known toxin sequences** (ricin, botulinum, etc.)
- **Virulence factors** from dangerous organisms

Let's implement a simplified screening workflow.

In [ ]:
# Simplified biosecurity screening database
# In practice, this would be a comprehensive curated database
THREAT_SIGNATURES = {
    'select_agents': [
        {'name': 'Botulinum toxin', 'organism': 'Clostridium botulinum',
         'motif': 'MPFVNKQFNYKDPV', 'category': 'Toxin'},
        {'name': 'Ricin A chain', 'organism': 'Ricinus communis',
         'motif': 'IFPKQYPIINFTAG', 'category': 'Toxin'},
        {'name': 'Anthrax protective antigen', 'organism': 'Bacillus anthracis',
         'motif': 'EVKQENRLLNESES', 'category': 'Virulence factor'},
        {'name': 'Ebola VP40', 'organism': 'Ebolavirus',
         'motif': 'MRRVILPTAPPEY', 'category': 'Structural protein'},
    ],
    'concern_organisms': [
        'Bacillus anthracis', 'Yersinia pestis', 'Variola major',
        'Francisella tularensis', 'Clostridium botulinum',
        'Ebolavirus', 'Marburg virus'
    ]
}

def screen_sequence(sequence, purpose, target_organism):
    """Screen a designed sequence against known threat signatures."""
    flags = []
    
    # 1. Check against known threat motifs
    for agent in THREAT_SIGNATURES['select_agents']:
        if agent['motif'] in sequence:
            flags.append({
                'level': 'CRITICAL',
                'type': 'Sequence match',
                'detail': f'Matches {agent["name"]} ({agent["organism"]})'
            })
    
    # 2. Check if target organism is a concern
    if target_organism in THREAT_SIGNATURES['concern_organisms']:
        flags.append({
            'level': 'WARNING',
            'type': 'Target organism',
            'detail': f'{target_organism} is a select agent / concern organism'
        })
    
    # 3. Flag if purpose is unclear
    benign_keywords = ['diagnostic', 'therapeutic', 'vaccine', 'research', 'sensor']
    if not any(kw in purpose.lower() for kw in benign_keywords):
        flags.append({
            'level': 'INFO',
            'type': 'Purpose review',
            'detail': 'Purpose does not match common benign applications'
        })
    
    # 4. Check sequence properties
    if len(sequence) < 20:
        flags.append({
            'level': 'INFO',
            'type': 'Short sequence',
            'detail': 'Very short sequences may be peptide toxins'
        })
    
    passed = all(f['level'] != 'CRITICAL' for f in flags)
    
    return {
        'passed': passed,
        'flags': flags,
        'recommendation': 'CLEARED for synthesis' if passed else 'BLOCKED — requires manual review'
    }

# Screen our Lassa binder
result = screen_sequence(
    sequence=example_seq,
    purpose='Diagnostic binder for Lassa GPC',
    target_organism='Lassa mammarenavirus'
)

print('=== BIOSECURITY SCREENING REPORT ===')
print(f'Result: {result["recommendation"]}')
print(f'Flags: {len(result["flags"])}')
for f in result['flags']:
    print(f'  [{f["level"]}] {f["type"]}: {f["detail"]}')
if not result['flags']:
    print('  No flags raised. Sequence is cleared.')

In [ ]:
# Demonstrate what a FLAGGED sequence looks like
flagged_result = screen_sequence(
    sequence='MPFVNKQFNYKDPVXXXXXXX',  # Contains botulinum motif
    purpose='Unknown',
    target_organism='Clostridium botulinum'
)

print('=== FLAGGED SCREENING EXAMPLE ===')
print(f'Result: {flagged_result["recommendation"]}')
for f in flagged_result['flags']:
    emoji = '\u274c' if f['level'] == 'CRITICAL' else '\u26a0\ufe0f' if f['level'] == 'WARNING' else '\u2139\ufe0f'
    print(f'  {emoji} [{f["level"]}] {f["type"]}: {f["detail"]}')

print()
print('In real DNA synthesis providers, this order would be BLOCKED')
print('and flagged for review by a biosafety officer.')

---
## Part 3: The Open vs. Closed Models Debate

### Discussion Framework

| Dimension | Open Models | Closed Models |
|-----------|------------|---------------|
| **Access** | Anyone can download & run | API-only, gated access |
| **Innovation** | Faster, community-driven | Controlled, corporate |
| **Safety** | Harder to prevent misuse | Can enforce screening |
| **Equity** | Available to African labs | May exclude under-resourced labs |
| **Examples** | ESMFold, ProteinMPNN | AlphaFold3 API, closed diffusion models |

### The African Perspective
For labs in Lagos, Nairobi, or Accra, **closed models create a dependency** on Western tech companies. Open models enable sovereignty but require local biosecurity infrastructure.

Let's quantify this trade-off.

In [ ]:
# Visualize the Open vs Closed trade-off
categories = ['Accessibility', 'Safety\nControl', 'Innovation\nSpeed', 
              'Equity for\nAfrican Labs', 'Reproducibility', 'Cost']

open_scores =   [9, 3, 9, 8, 9, 9]  # Open models
closed_scores = [3, 8, 5, 3, 4, 4]  # Closed models

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, open_scores, width, label='Open Models', color='#22c55e', alpha=0.8)
bars2 = ax.bar(x + width/2, closed_scores, width, label='Closed Models', color='#3b82f6', alpha=0.8)

ax.set_ylabel('Score (1-10)', fontsize=12)
ax.set_title('Open vs Closed Models: Trade-off Analysis for African Bio-Design Labs', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(int(bar.get_height())), ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(int(bar.get_height())), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('open_vs_closed.png', dpi=150)
plt.show()

print('\nKey takeaway: Open models score higher on equity and accessibility,')
print('but require local biosecurity frameworks to manage dual-use risk.')

---
## Part 4: Intellectual Property — Who Owns a Hallucinated Protein?

### The Question
If RFDiffusion "hallucinates" a novel protein backbone, and ProteinMPNN designs its sequence:
- Can you patent it?
- Does the University of Washington (RFDiffusion creators) have a claim?
- What about the training data — millions of natural proteins from PDB?

### Current Legal Landscape
1. **USPTO** (US): AI cannot be listed as an inventor, but the human who directed the AI can.
2. **EPO** (Europe): Similar stance — requires a human inventor.
3. **Nigeria** (NOTAP/Patents Act): Less clear — this is an opportunity for African IP leadership.

### Framework for Your Capstone
Document your **creative decisions** to strengthen any future IP claim:
- Choice of target (Lassa GPC)
- Epitope selection rationale
- Screening criteria and thresholds
- Intended application

In [ ]:
def generate_ip_documentation(design_params):
    """Generate an IP documentation template for a protein design."""
    doc = f"""
{'='*60}
INTELLECTUAL PROPERTY DOCUMENTATION
Lagos Bio-Design Bootcamp — Cohort 1
{'='*60}

Date: {datetime.datetime.now().strftime('%Y-%m-%d')}
Designer: {design_params.get('designer', 'TBD')}

1. INVENTIVE CONCEPT
   Target: {design_params.get('target', 'TBD')}
   Application: {design_params.get('application', 'TBD')}
   Novelty: {design_params.get('novelty', 'TBD')}

2. CREATIVE DECISIONS (strengthens IP claim)
   - Target selection rationale: {design_params.get('target_rationale', 'TBD')}
   - Epitope / binding site choice: {design_params.get('epitope_rationale', 'TBD')}
   - Screening criteria: {design_params.get('screening_criteria', 'TBD')}
   - Design parameters: {design_params.get('design_params_detail', 'TBD')}

3. AI TOOLS USED
   - Backbone generation: {design_params.get('backbone_tool', 'RFDiffusion')}
   - Sequence design: {design_params.get('sequence_tool', 'ProteinMPNN')}
   - Validation: {design_params.get('validation_tool', 'ESMFold')}

4. RESULTS SUMMARY
   - Candidates generated: {design_params.get('n_candidates', 'TBD')}
   - Top candidate pLDDT: {design_params.get('best_plddt', 'TBD')}
   - Top candidate RMSD: {design_params.get('best_rmsd', 'TBD')}

5. BIOSECURITY REVIEW
   - Screening passed: {design_params.get('screening_passed', 'TBD')}
   - Dual-use assessment: {design_params.get('dual_use', 'Low risk — diagnostic application')}
{'='*60}
"""
    return doc

# Generate documentation for the capstone project
ip_doc = generate_ip_documentation({
    'designer': 'Your Name Here',
    'target': 'Lassa virus Glycoprotein Complex (GPC), PDB: 5VK2',
    'application': 'Rapid diagnostic binder for Lassa fever detection',
    'novelty': 'First computationally designed binder targeting Lassa GPC GP1 subunit',
    'target_rationale': 'Lassa fever causes ~5,000 deaths/year in West Africa; no rapid diagnostic exists',
    'epitope_rationale': 'Conserved surface residues on GP1 receptor-binding domain',
    'screening_criteria': 'pLDDT > 80, RMSD < 2.0 Angstrom',
    'design_params_detail': '60-residue binders, 10 candidates, temp=0.1',
    'n_candidates': 10,
    'best_plddt': '92.3',
    'best_rmsd': '1.4 Angstrom',
    'screening_passed': 'Yes — no threat signatures detected'
})

print(ip_doc)

---
## Part 5: Regulatory Landscape for Biotech Startups in Lagos

### Key Regulatory Bodies
| Body | Role | Relevance |
|------|------|----------|
| **NAFDAC** | National Agency for Food and Drug Administration and Control | Approval of diagnostic devices |
| **NOTAP** | National Office for Technology Acquisition and Promotion | Technology transfer and IP |
| **NBMA** | National Biosafety Management Agency | Biosafety for GMOs/synthetic biology |
| **NCC** | Nigeria Centre for Disease Control | Disease surveillance, would be end-user |

### Path to Market for a Lassa Diagnostic
1. **In-silico validation** (This bootcamp) → Computational proof of concept
2. **Lab validation** → Express protein, test binding (SPR/ELISA)
3. **Prototype** → Integrate into lateral flow assay
4. **NAFDAC approval** → Clinical trials, regulatory submission
5. **Deployment** → Partner with NCDC for field deployment

In [ ]:
# Visualize the path to market
stages = [
    ('In-Silico\nDesign', 0.95, '3 months', 'Bootcamp'),
    ('Lab\nValidation', 0.60, '6-12 months', '$10-50K'),
    ('Prototype\nDevice', 0.35, '12-18 months', '$50-200K'),
    ('Regulatory\nApproval', 0.15, '18-36 months', '$200K-1M'),
    ('Field\nDeployment', 0.10, '36+ months', '$1M+')
]

fig, ax = plt.subplots(figsize=(12, 5))

x_positions = range(len(stages))
probabilities = [s[1] for s in stages]
colors = ['#22c55e', '#84cc16', '#eab308', '#f97316', '#ef4444']

bars = ax.bar(x_positions, probabilities, color=colors, alpha=0.8, width=0.6,
              edgecolor='white', linewidth=1.5)

# Add labels
for i, (stage, prob, timeline, cost) in enumerate(stages):
    ax.text(i, prob + 0.03, f'{prob*100:.0f}%', ha='center', fontsize=12, fontweight='bold')
    ax.text(i, -0.08, timeline, ha='center', fontsize=9, color='gray')
    ax.text(i, -0.14, cost, ha='center', fontsize=9, color='gray', style='italic')

ax.set_xticks(x_positions)
ax.set_xticklabels([s[0] for s in stages], fontsize=10)
ax.set_ylabel('Probability of Success', fontsize=12)
ax.set_title('Lassa Diagnostic Binder: Path to Market', fontsize=14)
ax.set_ylim(-0.2, 1.1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add "YOU ARE HERE" marker
ax.annotate('YOU ARE\nHERE', xy=(0, 0.95), xytext=(0.5, 1.05),
           arrowprops=dict(arrowstyle='->', color='#22c55e', lw=2),
           fontsize=11, fontweight='bold', color='#22c55e', ha='center')

plt.tight_layout()
plt.savefig('path_to_market.png', dpi=150)
plt.show()

---
## Part 6: Build Your Pitch Deck

### The Lagos Angel Network Pitch (5 slides)

Use this template to compile your capstone results into a startup pitch.

| Slide | Content | Time |
|-------|---------|------|
| 1. **Problem** | Lassa fever kills ~5,000/year; no rapid diagnostic | 1 min |
| 2. **Solution** | AI-designed binder protein for rapid test | 1 min |
| 3. **Science** | Your pLDDT, RMSD, screening results | 1 min |
| 4. **Market & Team** | West African market, your team | 1 min |
| 5. **Ask** | Funding for lab validation ($50K seed) | 1 min |

In [ ]:
def compile_pitch_metrics(capstone_results=None):
    """Compile key metrics for the pitch deck from capstone results.
    
    Replace the example values below with your actual Module 4 results.
    """
    # ============================================
    # EDIT THESE WITH YOUR ACTUAL CAPSTONE RESULTS
    # ============================================
    metrics = {
        'target': 'Lassa virus GPC (PDB: 5VK2)',
        'disease': 'Lassa fever',
        'annual_cases': '~300,000',
        'annual_deaths': '~5,000',
        'affected_region': 'West Africa (Nigeria, Sierra Leone, Liberia, Guinea)',
        'candidates_generated': 10,
        'candidates_passing': 3,
        'best_plddt': 92.3,
        'best_rmsd': 1.4,
        'binder_length': 60,
        'pipeline': 'RFDiffusion → ProteinMPNN → ESMFold',
        'seed_funding_ask': '$50,000',
        'next_step': 'Lab validation — protein expression + binding assay (SPR)',
    }
    
    print('=' * 60)
    print('PITCH DECK KEY METRICS')
    print('Lagos Bio-Design Bootcamp — Cohort 1')
    print('=' * 60)
    
    print(f'\n--- SLIDE 1: THE PROBLEM ---')
    print(f'Disease: {metrics["disease"]}')
    print(f'Annual cases: {metrics["annual_cases"]}')
    print(f'Annual deaths: {metrics["annual_deaths"]}')
    print(f'Region: {metrics["affected_region"]}')
    print(f'Current diagnostics: Slow, expensive, unavailable in rural areas')
    
    print(f'\n--- SLIDE 2: THE SOLUTION ---')
    print(f'AI-designed binder protein for rapid diagnostic test')
    print(f'Target: {metrics["target"]}')
    print(f'Pipeline: {metrics["pipeline"]}')
    
    print(f'\n--- SLIDE 3: THE SCIENCE ---')
    print(f'Candidates generated: {metrics["candidates_generated"]}')
    print(f'Candidates passing (pLDDT>80, RMSD<2.0): {metrics["candidates_passing"]}')
    print(f'Best pLDDT: {metrics["best_plddt"]} (threshold: 80)')
    print(f'Best RMSD: {metrics["best_rmsd"]} Å (threshold: 2.0)')
    print(f'Binder size: {metrics["binder_length"]} residues')
    
    print(f'\n--- SLIDE 4: MARKET & TEAM ---')
    print(f'TAM: Rapid diagnostic market in West Africa')
    print(f'Comparable: COVID rapid tests ($1-5 per test)')
    print(f'Team: Lagos Bio-Design Bootcamp graduates')
    
    print(f'\n--- SLIDE 5: THE ASK ---')
    print(f'Seed funding: {metrics["seed_funding_ask"]}')
    print(f'Next milestone: {metrics["next_step"]}')
    print(f'Timeline: 6-12 months to binding validation')
    print('=' * 60)
    
    return metrics

metrics = compile_pitch_metrics()

In [ ]:
# Generate a visual summary slide for the pitch
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: Pipeline
ax1 = axes[0]
pipeline_steps = ['Target\n(Lassa GPC)', 'RFDiffusion\n(Backbone)', 'ProteinMPNN\n(Sequence)',
                  'ESMFold\n(Validate)', 'Top 3\nCandidates']
y_pos = np.arange(len(pipeline_steps))
colors_pipe = ['#ef4444', '#f97316', '#eab308', '#22c55e', '#14b8a6']
ax1.barh(y_pos, [1]*5, color=colors_pipe, alpha=0.8, height=0.6)
for i, step in enumerate(pipeline_steps):
    ax1.text(0.5, i, step, ha='center', va='center', fontsize=10, fontweight='bold')
ax1.set_xlim(0, 1)
ax1.set_yticks([])
ax1.set_xticks([])
ax1.set_title('Design Pipeline', fontsize=13, fontweight='bold')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['bottom'].set_visible(False)
ax1.spines['left'].set_visible(False)

# Panel 2: Key metrics
ax2 = axes[1]
metric_names = ['pLDDT', 'RMSD (Å)', 'Candidates', 'Pass Rate']
metric_values = [92.3, 1.4, 10, 30]
metric_thresholds = [80, 2.0, None, None]
bar_colors = ['#22c55e', '#22c55e', '#3b82f6', '#3b82f6']
bars = ax2.bar(metric_names, metric_values, color=bar_colors, alpha=0.8, width=0.5)
for bar, val in zip(bars, metric_values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val) + ('%' if val == 30 else ''), ha='center', fontsize=11, fontweight='bold')
ax2.set_title('Key Results', fontsize=13, fontweight='bold')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Panel 3: Impact
ax3 = axes[2]
impact_data = {'Lassa\ncases/yr': 300000, 'Deaths/yr': 5000,
               'Rapid tests\navailable': 0, 'Our\ncandidates': 3}
colors_impact = ['#ef4444', '#ef4444', '#6b7280', '#22c55e']
ax3.bar(impact_data.keys(), impact_data.values(), color=colors_impact, alpha=0.8, width=0.5)
for i, (k, v) in enumerate(impact_data.items()):
    label = f'{v:,}' if v > 100 else str(v)
    ax3.text(i, v + 5000, label, ha='center', fontsize=11, fontweight='bold')
ax3.set_title('Impact: Lassa Fever', fontsize=13, fontweight='bold')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

plt.suptitle('Lagos Bio-Design Bootcamp — Capstone Pitch Summary', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('pitch_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print('Pitch summary saved to pitch_summary.png')

---
## Part 7: Ethics Self-Assessment Checklist

Complete this checklist before submitting your capstone. Every responsible bio-designer should be able to answer these questions.

In [ ]:
ethics_checklist = [
    ('Dual-Use Assessment', 'Could your designed protein be misused for harmful purposes?'),
    ('Biosecurity Screening', 'Have you screened your sequences against known threat databases?'),
    ('Informed Purpose', 'Is the intended application clearly documented and beneficial?'),
    ('Audit Trail', 'Can your design be traced back to you, your tools, and your rationale?'),
    ('Data Provenance', 'Are your training structures from legitimate, public sources (PDB)?'),
    ('Equity Consideration', 'Will the end product be accessible to those who need it most?'),
    ('Environmental Impact', 'Have you considered the ecological implications of your design?'),
    ('IP Transparency', 'Have you documented which AI tools contributed to the design?'),
    ('Regulatory Awareness', 'Do you know which regulatory bodies govern your application?'),
    ('Community Benefit', 'Does your design address a genuine need in the local context?'),
]

print('=' * 60)
print('ETHICS SELF-ASSESSMENT CHECKLIST')
print('Lagos Bio-Design Bootcamp — Module 5')
print('=' * 60)
print()

# ====================================
# EDIT: Set True/False for each item
# ====================================
responses = [True, True, True, True, True, True, False, True, True, True]

passed = 0
for i, ((title, question), response) in enumerate(zip(ethics_checklist, responses), 1):
    status = '\u2705' if response else '\u274c'
    print(f'{status} {i:2d}. {title}')
    print(f'     {question}')
    if response:
        passed += 1

print(f'\nScore: {passed}/{len(ethics_checklist)}')
if passed == len(ethics_checklist):
    print('\u2728 All checks passed. Your design meets ethical standards.')
elif passed >= 8:
    print('\u26a0\ufe0f Almost there. Address the unchecked items before submission.')
else:
    print('\u274c Several items need attention. Review your design process.')

---
## Summary

In this final module you explored the responsibilities that come with generative protein design:

1. **FoldMark Audit Trail** — Created traceable records for AI-designed proteins
2. **Biosecurity Screening** — Implemented sequence screening against threat databases
3. **Open vs. Closed Debate** — Analysed trade-offs from the African lab perspective
4. **Intellectual Property** — Documented creative decisions to support IP claims
5. **Regulatory Landscape** — Mapped the path from in-silico design to NAFDAC approval
6. **Pitch Compilation** — Compiled capstone results into a startup pitch
7. **Ethics Checklist** — Self-assessed responsible design practices

### Key Takeaways
- **Power comes with responsibility.** The same tools that design diagnostics could theoretically design threats.
- **Screening is non-negotiable.** Every sequence destined for synthesis must be screened.
- **Africa needs its own biosecurity infrastructure.** Don't wait for Western frameworks — build local ones.
- **Document everything.** Audit trails protect you and the community.

### Deliverable
Submit a **5-minute pitch video** or **PDF slide deck** covering:
1. The problem (Lassa fever)
2. Your solution (AI-designed binder)
3. Your science (pLDDT, RMSD, screening results)
4. Market & team
5. The ask (seed funding for lab validation)

---

**Congratulations on completing the Lagos Bio-Design Bootcamp!** 🎉

You now have the skills to design proteins computationally, validate them in-silico, and present them as viable biotech innovations. The future of African biotech starts with you.